# HAND flood inundation downstream of a dam

This notebook runs the whole iGuide HAND (Height Above Nearest Drainage) pipeline for one dam. It goes from the dam's coordinates to a flood depth map:

| Step | What it does | Code |
|---|---|---|
| 0 | Builds the dam folder: dam point, `stage.txt`, NHDPlus HR HU4 data (downloaded) | `dam_setup.py` |
| 1 | Walks the HUC12s downstream of the dam and clips the NHDPlus rasters | `HUC12BoundingBoxCalculator` (`dam_catchments.py`) |
| 2 | TauDEM terrain analysis: HAND grid, stream network split into reaches | `HandDamWorkflow` (`dam_hand.py`) |
| 3 | Synthetic rating curves for each reach | `DamHydroProp` (`dam_hydroprop.py`) |
| 4 | Flow scenario(s): your `forecast.csv`, or a uniform flow on every reach | `write_uniform_forecast` (`dam_inundation.py`) |
| 5 | Inundation depth for each scenario | `DamInundationAnalysis` (`dam_inundation.py`) |

**Example:** Mountain Dell Dam (Parleys Creek, Salt Lake City, Utah). It is in HU4 1602 (Great Salt Lake). With 2 km reaches, the study reach ends where the creek enters Great Salt Lake, about 60 km below the dam.

**To run another dam,** edit only the *Parameters* cell. Set `DAM_LABEL` (the dam's name) and its `DAM_LAT` / `DAM_LON`, then run all cells. The notebook finds the HU4 and downloads its NHDPlus HR data. The raster zip is 1–6 GB, so allow about 3× that in free disk space.

**Requirements:** TauDEM ≥ 5.4.0 and `mpiexec` installed on this machine, and the `iguide-hand` kernel (see the README section *Running in JupyterHub*). Once the data is in place, the pipeline takes about 3 minutes for Mountain Dell.

## Parameters
Edit this cell for your dam. Everything below it runs unchanged.

In [ ]:
# --- Dam ---------------------------------------------------------------------
DAM_LABEL = "Mountain Dell Dam"      # dam name, stored in the dam shapefile and shown in plot titles
DAM_NAME = None                      # folder name for data/<DAM_NAME> and outputs/<DAM_NAME>;
                                     # None = from DAM_LABEL ("Mountain Dell Dam" -> mountain_dell)
DAM_LAT, DAM_LON = 40.750594, -111.721088   # dam location, WGS84 decimal degrees. Best on the
                                     # channel just below the embankment: NID coordinates often
                                     # fall in the reservoir, where the DEM is flat
HU4 = None                           # 4-digit HUC containing the dam; None = look it up

# --- Locations ---------------------------------------------------------------
DATA_ROOT = None                     # None = <repo>/data
OUTPUT_ROOT = None                   # None = <repo>/outputs
NHD_CACHE_DIR = None                 # shared folder for NHDPlus HR downloads (e.g. DATA_ROOT),
                                     # so dams in the same HU4 download it once; None = the dam folder
TAUDEM_PATH = "~/taudem"             # folder with the TauDEM executables
NP = 4                               # MPI processes; reach IDs depend on it, so keep it fixed

# --- Step 1: downstream catchments -------------------------------------------
MAX_DISTANCE_KM = 100                # stop the HUC12 walk after this distance
BUFFER_M = 4000                      # buffer around the HUC12s when clipping rasters

# --- Step 2: HAND and reaches ------------------------------------------------
SEGMENT_LENGTH_M = 2000              # reach length; None = use Outlets2_utm.shp in the dam folder
MAX_REACH_DISTANCE_M = None          # end the reaches this far below the dam; None = stop at the
                                     # first large lake/reservoir, else at the end of the stream
MIN_WATERBODY_AREA_KM2 = 10          # smallest lake/reservoir that ends the reaches
STREAM_THRESHOLD = 10000             # contributing area (cells) that starts a stream
DAM_MOVE_DISTANCE_CELLS = 200        # how far (cells) the dam point may move down its flow path
                                     # to reach the stream grid
MANNING_N = 0.05                     # Manning's n for every reach

# --- Step 3: rating curves ---------------------------------------------------
STAGES = None                        # stage heights (m) for a new stage.txt; None = 0.1 ... 40 m

# --- Steps 4-5: flow scenarios -----------------------------------------------
FORECAST_FILE = None                 # your forecast CSV (Id, flow_m3s) matching this run's reaches
UNIFORM_FLOWS_M3S = [4000]           # used when FORECAST_FILE is None: one scenario per flow
DETAIL_HALF_WIDTH_M = 3000           # half-width of the full-resolution depth close-ups

## Setup

In [ ]:
import logging
import os
import shutil
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
import shapely.ops
from IPython.display import Markdown, display
from rasterio.windows import Window, from_bounds

# Repo root: the folder with pyproject.toml, at or above the notebook's folder
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))  # the pipeline scripts import each other as top-level modules

from dam_catchments import HUC12BoundingBoxCalculator
from dam_hand import HandDamWorkflow
from dam_hydroprop import DamHydroProp
from dam_inundation import DamInundationAnalysis, write_uniform_forecast
from dam_setup import dam_slug, download_nhdplus_hr, find_hu4_for_point, write_dam_point, write_stage_file
from utils import find_dam_shapefile

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    stream=sys.stdout, force=True)
# TauDEM's console output is long; only show TauDEM failures. Set to logging.INFO to see it all.
logging.getLogger("utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="pyogrio")  # shapefile field-name / width notices

DATA_ROOT = Path(DATA_ROOT or REPO / "data").expanduser().resolve()
OUTPUT_ROOT = Path(OUTPUT_ROOT or REPO / "outputs").expanduser().resolve()
DAM_NAME = DAM_NAME or dam_slug(DAM_LABEL)
DAM_DIR = DATA_ROOT / DAM_NAME
OUT_DIR = OUTPUT_ROOT / DAM_NAME
TAUDEM_PATH = str(Path(os.path.expandvars(TAUDEM_PATH)).expanduser().resolve())
print(f"Dam:     {DAM_LABEL} ({DAM_NAME})\nRepo:    {REPO}\nDam dir: {DAM_DIR}\nOutputs: {OUT_DIR}")

# TauDEM must be installed; check the tools this notebook uses
os.environ["PATH"] = f"{TAUDEM_PATH}{os.pathsep}{os.environ['PATH']}"
tools = ["mpiexec", "pitremove", "d8flowdir", "dinfdistdown", "streamnet", "catchhydrogeo", "inundepth"]
missing = [t for t in tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(f"Not found on PATH (TAUDEM_PATH={TAUDEM_PATH}): {missing}. "
                       "Install TauDEM >= 5.4.0 and MPI, or fix TAUDEM_PATH.")
print("TauDEM found:", shutil.which("pitremove"))


def run_stage(name, fn):
    # The pipeline stages call sys.exit(1) on failure; turn that into a normal error
    try:
        return fn()
    except SystemExit as e:
        raise RuntimeError(f"{name} failed (exit code {e.code}); see the log above") from None


def read_raster(path, max_size=1500, bounds=None):
    # Read band 1 (optionally only within bounds = (minx, miny, maxx, maxy)),
    # decimated so the longer side is at most max_size pixels
    with rasterio.open(path) as src:
        window = (from_bounds(*bounds, transform=src.transform)
                  .intersection(Window(0, 0, src.width, src.height))
                  .round_offsets().round_lengths() if bounds else Window(0, 0, src.width, src.height))
        scale = max(1, max(window.width, window.height) / max_size)
        shape = (max(1, int(window.height / scale)), max(1, int(window.width / scale)))
        data = src.read(1, window=window, out_shape=shape, masked=True).astype(float)
        left, bottom, right, top = src.window_bounds(window)
        return data, (left, right, bottom, top), src.crs


def hillshade(dem, azimuth=315, altitude=45):
    dy, dx = np.gradient(dem.filled(np.nan))
    slope = np.pi / 2 - np.arctan(np.hypot(dx, dy))
    aspect = np.arctan2(-dx, dy)
    az, alt = np.radians(azimuth), np.radians(altitude)
    return np.sin(alt) * np.sin(slope) + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)


def plot_layer(ax, path, crs, **kwargs):
    # Plot a vector file reprojected to crs
    gpd.read_file(path).to_crs(crs).plot(ax=ax, **kwargs)

## Step 0: prepare the dam folder

This step makes `data/<DAM_NAME>/` self-contained:
- the dam point (`<DAM_NAME>_dam_location.shp`);
- `stage.txt`;
- the NHDPlus HR GDB and raster folder for the dam's HU4.

Files that already exist are kept. To start over, delete them first. If you change `DAM_LAT` / `DAM_LON` after a run, delete the old `*_dam_location.shp` first. Otherwise this step stops with an error rather than silently using the old point.

In [ ]:
if HU4 is None:
    HU4 = find_hu4_for_point(DAM_LAT, DAM_LON)
print("HU4:", HU4)

write_dam_point(DAM_DIR, DAM_LAT, DAM_LON, DAM_NAME, label=DAM_LABEL)
write_stage_file(DAM_DIR, STAGES)
HUC12_GDB, RASTER_DIR = download_nhdplus_hr(HU4, DAM_DIR, cache_dir=NHD_CACHE_DIR)
DAM_SHP = find_dam_shapefile(DAM_DIR)

print(f"\nDam shapefile: {DAM_SHP}\nNHDPlus GDB:   {HUC12_GDB}\nRaster folder: {RASTER_DIR}")
print("\nDam folder contents:")
for p in sorted(DAM_DIR.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))

## Step 1: downstream HUC12s and clipped rasters

The walk follows the HUC12 `tohuc` links downstream from the dam. It stops at `MAX_DISTANCE_KM`, at a terminal basin, or at the edge of the HU4. The NHDPlus rasters are then clipped to the HUC12s plus `BUFFER_M`.

In [ ]:
calc = HUC12BoundingBoxCalculator(
    dam_shapefile=DAM_SHP,
    huc12_gdb_path=HUC12_GDB,
    ref_projection_raster_path=RASTER_DIR / "hydrodem.tif",
    output_dir=OUT_DIR,
    raster_dir=RASTER_DIR,
    buffer_distance=BUFFER_M,
)
calc.max_distance_km = MAX_DISTANCE_KM
bbox = calc.run_analysis(clip_rasters=True)

print(f"\nHUC12s downstream: {len(calc.downstream_huc12s)}, distance {calc.total_distance:.1f} km")
print(f"Stopped: {calc.stop_reason}")
print(f"Clipped rasters: {len(calc.clipped_rasters)} ({len(calc.failed_rasters)} failed) in {calc.clipped_dir}")
if calc.stop_reason and calc.stop_reason.startswith("left the HU4"):
    display(Markdown("**Warning:** the river leaves this HU4 before the distance limit, so the study "
                     "area is cut off at the HU4 boundary. Only one HU4 is supported."))

In [ ]:
huc12s = gpd.read_file(OUT_DIR / "downstream_huc12s_from_dam.shp")
fig, ax = plt.subplots(figsize=(8, 6))
huc12s.plot(ax=ax, column="name", cmap="tab20", edgecolor="k", alpha=0.6, legend=True,
            legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1), "fontsize": 8})
plot_layer(ax, DAM_SHP, huc12s.crs, color="red", marker="^", markersize=120, zorder=5)
ax.set_title(f"{DAM_LABEL}: downstream HUC12s (dam = red triangle)")
ax.set_axis_off()

## Step 2: HAND and reaches (TauDEM)

This step runs TauDEM on the clipped rasters: flow directions, contributing area, the stream grid below the dam, and HAND. It then splits the main stem into reaches.
- With `SEGMENT_LENGTH_M`, reaches are about that long. They end just upstream of the first large lake or reservoir in the NHDPlus `NHDWaterbody` layer, or at `MAX_REACH_DISTANCE_M`.
- With `SEGMENT_LENGTH_M = None`, reaches break at the points in `Outlets2_utm.shp` in the dam folder.

This step takes a few minutes. The reach IDs (`LINKNO` / `Id`) belong to this run: they change if the outlets or `NP` change.

In [ ]:
hand = HandDamWorkflow(
    dam_dir=DAM_DIR,
    taudem_path=TAUDEM_PATH,
    output_dir=OUT_DIR,
    dam_shapefile=DAM_SHP,
    segment_length=SEGMENT_LENGTH_M,
    max_distance=MAX_REACH_DISTANCE_M,
    min_waterbody_area_km2=MIN_WATERBODY_AREA_KM2,
    stream_threshold=STREAM_THRESHOLD,
    dam_move_distance=DAM_MOVE_DISTANCE_CELLS,
    num_processes=NP,
    manning_n=MANNING_N,
)
run_stage("HAND workflow", hand.run_workflow)

snap = hand.dam_snap
print(f"\nDam point moved {snap['moved_m']:g} m; contributing area there "
      f"{snap['contributing_area_km2']:g} km²; on the stream grid: {snap['on_stream']}")
if not snap["on_stream"]:
    display(Markdown(
        "**Warning:** the dam point did not reach the stream grid, so the main stem starts on a "
        "hillslope path and results near the dam are unreliable. Move `DAM_LAT` / `DAM_LON` onto "
        "the channel just below the dam, or raise `DAM_MOVE_DISTANCE_CELLS`, and rerun."))
if snap["in_waterbody"] and "snapped dam outlet" in snap["in_waterbody"]:
    display(Markdown(
        f"**Warning:** {snap['in_waterbody']}. The DEM is flat across the reservoir, so the main "
        "stem may leave it by the wrong side instead of through the dam. Set `DAM_LAT` / `DAM_LON` "
        "to a point on the channel just below the dam embankment, delete the old "
        "`*_dam_location.shp` in the dam folder, and rerun."))

In [ ]:
catchlist = pd.read_csv(OUT_DIR / "catchlist.csv", skipinitialspace=True)
print(f"{len(catchlist)} reaches, total length {catchlist['Length'].sum() / 1000:.1f} km")
display(catchlist.describe().loc[["min", "50%", "max"], ["Slope", "Length"]])

hand_grid, extent, crs = read_raster(OUT_DIR / "hand.tif")
dem, dem_extent, _ = read_raster(OUT_DIR / "fel.tif")
net = gpd.read_file(OUT_DIR / "net2.shp").to_crs(crs)
reaches = net[net["LINKNO"].isin(catchlist["Id"])]

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(hillshade(dem), extent=dem_extent, cmap="gray", alpha=0.5)
# Colour only the low HAND values that floods can reach; higher ground is left clear
im = ax.imshow(np.ma.masked_greater(hand_grid, 30), extent=extent, cmap="viridis_r",
               vmin=0, vmax=30, alpha=0.8)
net.plot(ax=ax, color="lightblue", linewidth=0.6)
reaches.plot(ax=ax, color="blue", linewidth=1.5)
outlets = OUT_DIR / "Outlets2_geo.shp"
if outlets.exists():
    plot_layer(ax, outlets, crs, color="orange", markersize=15, zorder=4)
plot_layer(ax, DAM_SHP, crs, color="red", marker="^", markersize=120, zorder=5)
fig.colorbar(im, ax=ax, shrink=0.7, label="HAND (m), shown up to 30 m")
ax.set_title(f"{DAM_LABEL}: HAND, study reaches (blue) and reach outlets (orange)")
ax.set_axis_off()

## Step 3: synthetic rating curves

For each reach and each stage height in `stage.txt`, TauDEM `catchhydrogeo` computes the reach's wetted area, volume, cross-section and hydraulic radius. Manning's equation then gives the discharge. The resulting table, `hydroprop.csv`, depends only on the terrain and the reaches. It is computed once and serves any number of flow scenarios.

In [ ]:
run_stage("Hydraulic properties", DamHydroProp(taudem_path=TAUDEM_PATH, output_dir=OUT_DIR, np=NP).run)

hydroprop = pd.read_csv(OUT_DIR / "hydroprop.csv", skipinitialspace=True)
# Plot about six reaches spread along the main stem
ids = catchlist["Id"].iloc[np.linspace(0, len(catchlist) - 1, min(6, len(catchlist))).astype(int)]
fig, ax = plt.subplots(figsize=(7, 5))
for rid in ids:
    curve = hydroprop[hydroprop["Id"] == rid]
    ax.plot(curve["Flow_m3s"], curve["Stage_m"], marker=".", label=f"reach {rid}")
ax.set_xscale("log")
ax.set_xlabel("Discharge (m³/s)")
ax.set_ylabel("Stage (m)")
ax.set_title("Synthetic rating curves")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

## Step 4: flow scenarios

A forecast is a CSV with one row per reach: `Id, flow_m3s`. The `Id` values must match **this run's** `catchlist.csv`, because reach IDs change whenever the outlets change. If you have a forecast that matches, set `FORECAST_FILE`. Otherwise this step writes one uniform-flow forecast per value in `UNIFORM_FLOWS_M3S`.

In [ ]:
if FORECAST_FILE is not None:
    forecasts = {Path(FORECAST_FILE).stem: Path(FORECAST_FILE).expanduser().resolve()}
else:
    forecasts = {f"{q:g}": write_uniform_forecast(OUT_DIR, q) for q in UNIFORM_FLOWS_M3S}
for tag, path in forecasts.items():
    print(f"{tag}: {path}")

## Step 5: inundation depth

For each scenario, TauDEM `inundepth` looks up each reach's water level on its rating curve. It then maps depth as that level minus HAND in the reach's catchment. All scenarios reuse `hydroprop.csv`.

In [ ]:
results = {}
for tag, forecast in forecasts.items():
    inun = DamInundationAnalysis(
        taudem_path=TAUDEM_PATH,
        output_dir=OUT_DIR,
        forecast_file=forecast,
        inundation_raster=OUT_DIR / f"inundepth_{tag}.tif",
        depth_table=OUT_DIR / f"depths_{tag}.csv",
        np=NP,
    )
    run_stage(f"Inundation ({tag})", inun.run_analysis)
    results[tag] = (inun.inundation_raster, inun.depth_table)

In [ ]:
for tag, (raster, table) in results.items():
    depth, extent, crs = read_raster(raster)
    depth = np.ma.masked_less_equal(depth, 0)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(hillshade(dem), extent=dem_extent, cmap="gray")
    im = ax.imshow(depth, extent=extent, cmap="Blues", vmin=0,
                   vmax=float(np.nanpercentile(depth.compressed(), 98)) if depth.count() else 1)
    reaches.to_crs(crs).plot(ax=ax, color="navy", linewidth=0.5)
    plot_layer(ax, DAM_SHP, crs, color="red", marker="^", markersize=120, zorder=5)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Inundation depth (m)")
    ax.set_title(f"{DAM_LABEL}: inundation depth, scenario {tag}")
    ax.set_axis_off()
    plt.show()

    # Full-resolution close-ups at 1/4, 1/2 and 3/4 of the length of the study reaches;
    # on a long reach the floodplain is too narrow to see in the overview
    stem = reaches.to_crs(crs).geometry.union_all()
    stem = shapely.ops.linemerge(stem) if stem.geom_type == "MultiLineString" else stem
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, frac in zip(axes, [0.25, 0.5, 0.75]):
        c = stem.interpolate(frac, normalized=True)
        box = (c.x - DETAIL_HALF_WIDTH_M, c.y - DETAIL_HALF_WIDTH_M,
               c.x + DETAIL_HALF_WIDTH_M, c.y + DETAIL_HALF_WIDTH_M)
        z, z_extent, _ = read_raster(OUT_DIR / "fel.tif", bounds=box)
        d, d_extent, _ = read_raster(raster, bounds=box)
        ax.imshow(hillshade(z), extent=z_extent, cmap="gray")
        ax.imshow(np.ma.masked_less_equal(d, 0), extent=d_extent, cmap="Blues",
                  vmin=im.norm.vmin, vmax=im.norm.vmax)
        ax.set_title(f"{frac:.0%} along the study reaches")
        ax.set_axis_off()
    fig.suptitle(f"Close-ups, {2 * DETAIL_HALF_WIDTH_M / 1000:g} km across")
    plt.show()

    depths = pd.read_csv(table, skipinitialspace=True)
    capped = int((depths["InunRatio"] >= 1).sum())
    print(f"Scenario {tag}: {len(depths)} reaches, flooded area "
          f"{depths['InunArea_m2'].sum() / 1e6:.1f} km², max reach depth {depths['depth'].max():.2f} m")
    if capped:
        print(f"  {capped} reach(es) have InunRatio = 1: the water fills the whole catchment, "
              "so the catchment, not the channel, caps the level and that depth is not reliable.")
    display(depths.head(10))

## Outputs

All files are in `outputs/<DAM_NAME>/`:

| File | What it is |
|---|---|
| `downstream_huc12s_from_dam.shp` | HUC12s downstream of the dam (study area) |
| `processing_report.txt` | Catchment walk summary, including why it stopped |
| `run_info.json` | Input paths recorded by step 1 |
| `net2.shp` | Stream network, split into reaches |
| `Outlets2_geo.shp` | Reach outlet points |
| `hand.tif` | Height Above Nearest Drainage |
| `catchlist.csv` | Reaches: ID, slope, length, Manning's n |
| `hydroprop.csv` | Rating curve for each reach (stage, area, volume, discharge) |
| `forecast_*.csv` | Flow scenarios |
| `inundepth_<scenario>.tif` | Flood depth map |
| `depths_<scenario>.csv` | Water level for each reach (`InunRatio = 1`: depth capped by the catchment) |

To try another flow, change `UNIFORM_FLOWS_M3S` (or `FORECAST_FILE`) and rerun only steps 4 and 5.

In [ ]:
for name in ["downstream_huc12s_from_dam.shp", "processing_report.txt", "net2.shp", "hand.tif",
             "catchlist.csv", "hydroprop.csv"] + [p.name for r in results.values() for p in r]:
    p = OUT_DIR / name
    print(f"{name:40s} {p.stat().st_size / 1e6:8.2f} MB" if p.exists() else f"{name:40s} missing")